# Movie Market Analysis using PySpark

**Dataset:** IMDB Movie Data (2006–2016)  
**Tools:** Apache Spark (PySpark), Databricks Community Edition  
**Goal:** Analyze movie market trends including revenue patterns, genre performance, ratings distribution, and year-wise box office performance using distributed data processing.

> **Note:** This notebook was built and executed on Databricks Community Edition using a Spark cluster. The file paths reference the Databricks FileStore. To run locally, replace the CSV path with your local file path and initialize a SparkSession manually.

---
## Section 1 — Data Loading
Load the IMDB movie dataset from the Databricks FileStore into a Spark DataFrame. We use `inferSchema=True` to automatically detect column types, and handle multiline fields and quoted strings that are common in CSV exports.

In [ ]:
df = spark.read.csv(
    "/FileStore/tables/IMDB_Movie_Data-2.csv",
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"',
    quote='"',
    ignoreLeadingWhiteSpace=True,
    ignoreTrailingWhiteSpace=True
)
df.display()

---
## Section 2 — Dataset Overview
Before any analysis, we check the basic shape of the dataset — how many rows and columns we are working with. This is the first step of any EDA (Exploratory Data Analysis).

In [ ]:
# Number of records (rows) and columns
row_count = df.count()
column_count = len(df.columns)

print("Number of records (rows):", row_count)
print("Number of columns:", column_count)
print("Column names:", df.columns)

---
## Section 3 — Data Type Casting
Even though `inferSchema=True` attempts to detect types automatically, we explicitly cast each numeric column to its correct type. This ensures there are no silent type mismatches during aggregations and calculations later in the analysis.

In [ ]:
from pyspark.sql.functions import col

df = df.withColumn("Year", col("Year").cast("int")) \
       .withColumn("Runtime (Minutes)", col("Runtime (Minutes)").cast("int")) \
       .withColumn("Rating", col("Rating").cast("float")) \
       .withColumn("Votes", col("Votes").cast("int")) \
       .withColumn("Revenue (Millions)", col("Revenue (Millions)").cast("float")) \
       .withColumn("Metascore", col("Metascore").cast("int"))

# Verify schema after casting
df.printSchema()

---
## Section 4 — Duplicate Check
We check for duplicate rows before proceeding with analysis. Duplicates can silently skew aggregations like total revenue or average ratings, making results inaccurate.

In [ ]:
duplicate_count = df.count() - df.dropDuplicates().count()
print("Number of duplicate rows:", duplicate_count)

if duplicate_count > 0:
    df = df.dropDuplicates()
    print("Duplicates removed. New row count:", df.count())
else:
    print("No duplicates found. Dataset is clean.")

---
## Section 5 — Null Value Analysis
We check every column for missing values using both `isnull()` and `isnan()`. Using both is important because float columns can contain `NaN` (Not a Number) which `isnull()` alone would miss.

In [ ]:
from pyspark.sql.functions import col, isnan, isnull, count, when

# Count null or NaN values for each column
missing_counts = df.select([
    count(when(isnull(c) | isnan(c), c)).alias(c)
    for c in df.columns
])

missing_counts.show()

---
## Section 6 — Handling Missing Values
The `Revenue (Millions)` and `Metascore` columns contain missing values. We fill these with their respective column means — a standard imputation strategy that preserves the overall distribution without introducing extreme values or losing rows.

In [ ]:
from pyspark.sql.functions import mean

# Calculate mean for Revenue and Metascore
revenue_mean = df.select(mean("Revenue (Millions)")).collect()[0][0]
metascore_mean = df.select(mean("Metascore")).collect()[0][0]

print(f"Revenue mean used for imputation: {round(revenue_mean, 2)}")
print(f"Metascore mean used for imputation: {round(metascore_mean, 2)}")

# Fill missing values with their respective means
df = df.fillna({
    "Revenue (Millions)": revenue_mean,
    "Metascore": metascore_mean
})

df.display()

---
## Section 7 — Rating Standardisation
We round all ratings to one decimal place for consistency. This avoids minor floating point differences like 7.800001 vs 7.8 from causing issues in filters and groupings later.

In [ ]:
from pyspark.sql.functions import round

df = df.withColumn("Rating", round(df["Rating"], 1))
df.display()

---
## Section 8 — Top 3 Movies Per Year by Revenue (Window Function)
We use a **Window Function** — one of PySpark's most powerful features — to rank movies within each year by revenue. Window functions perform calculations across a group of rows (a "window") without collapsing them into a single result like `groupBy` does.

Using `rank()` instead of `row_number()` correctly handles ties — if two movies have identical revenue in the same year, both receive the same rank.

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, col

# Define window: partition by Year, order by Revenue descending
windowSpec = Window.partitionBy("Year").orderBy(col("Revenue (Millions)").desc())

# Add rank column within each year's window
df_ranked = df.withColumn("rank", rank().over(windowSpec))

# Filter top 3 per year
top_3_movies = df_ranked.filter(col("rank") <= 3)

# Show results ordered clearly
top_3_movies.select("Year", "Title", "Revenue (Millions)", "rank") \
            .orderBy("Year", "rank") \
            .display()

---
## Section 9 — Highest and Lowest Rated Movies
We find the movies with the maximum and minimum IMDB ratings. Using `max()` and `min()` functions and then filtering by those values correctly captures all tied movies — unlike sorting and taking the first row which would silently miss ties.

In [ ]:
from pyspark.sql.functions import col, max, min

# Find max and min rating values
max_rating = df.select(max("Rating")).first()[0]
min_rating = df.select(min("Rating")).first()[0]

print(f"Highest Rating: {max_rating}")
print(f"Lowest Rating: {min_rating}")

# Filter all movies matching max and min (handles ties correctly)
highest_rated = df.filter(col("Rating") == max_rating)
lowest_rated = df.filter(col("Rating") == min_rating)

print("\nMovie(s) with Highest Rating:")
highest_rated.select("Title", "Rating", "Year", "Genre").display()

print("\nMovie(s) with Lowest Rating:")
lowest_rated.select("Title", "Rating", "Year", "Genre").display()

---
## Section 10 — Year-wise Total Revenue
We aggregate total box office revenue by year to identify which years were strongest for the movie industry. This gives a macro-level view of the market trend across the 2006–2016 period.

In [ ]:
from pyspark.sql.functions import sum, round

yearwise_revenue = df.groupBy("Year").agg(
    round(sum("Revenue (Millions)"), 2).alias("Total_Revenue_Millions")
).orderBy("Year")

yearwise_revenue.show()

---
## Section 11 — Top 3 Highest Revenue Movies of 2016
A focused slice of the data — looking at the top performing films specifically in 2016, the most recent complete year in the dataset.

In [ ]:
from pyspark.sql.functions import col

top_3_2016 = df.filter(col("Year") == 2016) \
               .orderBy(col("Revenue (Millions)").desc()) \
               .select("Title", "Revenue (Millions)", "Rating", "Genre") \
               .limit(3)

print("Top 3 Highest Revenue Movies of 2016:")
top_3_2016.show()

---
## Section 12 — Action, Adventure, Fantasy Genre Analysis
We filter for movies tagged exactly as 'Action,Adventure,Fantasy' — a blockbuster combination that typically dominates box office charts. This helps understand how this specific genre cluster performs across the dataset.

In [ ]:
from pyspark.sql.functions import col

action_adventure_fantasy = df.filter(col("Genre") == "Action,Adventure,Fantasy")

print(f"Total Action,Adventure,Fantasy movies: {action_adventure_fantasy.count()}")
action_adventure_fantasy.select("Title", "Year", "Revenue (Millions)", "Rating").display()

---
## Section 13 — Lowest Revenue Movie of 2009
We identify which movie generated the lowest box office revenue in 2009 and who directed it. We use `min()` first and then filter — this correctly handles the case where multiple movies share the lowest revenue value.

In [ ]:
from pyspark.sql.functions import col, min

# Step 1: Filter for 2009
df_2009 = df.filter(col("Year") == 2009)

# Step 2: Find the minimum revenue in 2009
min_revenue_2009 = df_2009.select(min("Revenue (Millions)")).first()[0]
print(f"Lowest revenue in 2009: ${min_revenue_2009}M")

# Step 3: Filter for movie(s) with that minimum revenue
lowest_revenue_movie_2009 = df_2009.filter(col("Revenue (Millions)") == min_revenue_2009)

# Step 4: Show the director and movie details
lowest_revenue_movie_2009.select("Title", "Director", "Revenue (Millions)", "Rating").show()

---
## Section 14 — Genre-wise Total Revenue Analysis
We explode the Genre column (since one movie can belong to multiple genres) and calculate total revenue per genre. This reveals which genres have historically generated the most and least box office revenue — a key insight for production and investment decisions.

In [ ]:
from pyspark.sql.functions import col, explode, split, sum, round

# Split comma-separated genres into individual rows
df_genres = df.withColumn("Genre", explode(split(col("Genre"), ",")))

# Trim whitespace from genre names after splitting
from pyspark.sql.functions import trim
df_genres = df_genres.withColumn("Genre", trim(col("Genre")))

# Group by genre and calculate total revenue
genre_revenue = df_genres.groupBy("Genre").agg(
    round(sum("Revenue (Millions)"), 2).alias("Total_Revenue_Millions")
)

# Sort by revenue descending
sorted_genres = genre_revenue.orderBy(col("Total_Revenue_Millions").desc())

# Get highest and lowest revenue genres
highest_revenue_genre = sorted_genres.first()
lowest_revenue_genre = sorted_genres.tail(1)[0]

print(f"Genre with HIGHEST total revenue: {highest_revenue_genre['Genre']} — ${highest_revenue_genre['Total_Revenue_Millions']}M")
print(f"Genre with LOWEST total revenue:  {lowest_revenue_genre['Genre']} — ${lowest_revenue_genre['Total_Revenue_Millions']}M")

print("\nFull genre revenue breakdown:")
sorted_genres.show(truncate=False)

---
## Section 15 — Top 10 Movies Per Year (Spark SQL via Temp View)
We demonstrate Spark SQL integration by creating a temporary view and registering our ranked DataFrame into it. This shows how PySpark bridges the gap between DataFrame API and SQL — a workflow commonly used in production Databricks environments.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Define window spec: partition by Year, order by Revenue descending
windowSpec = Window.partitionBy("Year").orderBy(F.col("Revenue (Millions)").desc())

# Rank movies by revenue within each year
df_ranked_top10 = df.withColumn("rank", F.rank().over(windowSpec))

# Filter top 10 movies per year
top_10_movies = df_ranked_top10.filter(F.col("rank") <= 10)

# Register as a Spark SQL temporary view
top_10_movies.createOrReplaceTempView("Top_10_Movies_Per_Year")
print("Temporary view 'Top_10_Movies_Per_Year' created successfully.")

# Query it using Spark SQL to demonstrate SQL integration
spark.sql("""
    SELECT Year, Title, `Revenue (Millions)`, rank
    FROM Top_10_Movies_Per_Year
    ORDER BY Year, rank
""").display()

---
## Summary — Key Findings

This analysis of IMDB movie data (2006–2016) using PySpark on Databricks revealed the following:

- **Revenue growth:** Total box office revenue shows a clear upward trend across the decade, with peak years in 2015–2016
- **Genre dominance:** Action and Adventure genres consistently generate the highest total revenue, while niche genres like Documentary and Music generate significantly less
- **Rating vs Revenue:** High ratings do not always correlate with high revenue — some critically acclaimed films underperformed commercially
- **Window functions** proved essential for year-wise competitive analysis, enabling rankings that preserve ties accurately
- **Spark SQL integration** via temp views demonstrates production-ready workflow patterns used in enterprise Databricks environments

**Technical concepts demonstrated:**
- PySpark DataFrame API — loading, casting, filtering, groupBy, aggregation
- Window Functions — `rank()`, `row_number()`, `partitionBy()`
- Null handling — `isnull()` + `isnan()` combined, mean imputation
- Data exploding — splitting multi-value columns into individual rows
- Spark SQL — `createOrReplaceTempView()`, SQL queries on DataFrames
- Databricks environment — FileStore, cluster-based execution, `.display()` method